<p align="center">
    <span style="font-size:2.5em; font-weight:bold;">
        eFleetPlan - Optimal infrastructure and fleet operation of electric LCV
    </span>
</p>

<p align="center">
    <span style="font-size:1.5em; font-weight:bold;">
        Carolina Gil Ribeiro, Jagruti Thakur
    </span>
</p>

## 0. Importing dependencies

In [1]:
from logging import config

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from glob import glob
import time
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from config.config_loader_optimisation import load_opt_config, RunOptConfig, InfrastructureConfig
from src.efleetplan._2_co_optimisation.co_optimisation import optimisation, save_results
from src.efleetplan._2_co_optimisation.optimisation_graphs import graph_vehicles, plot_summary_table, process_folder, graph_number_of_chargers_by_schedules, graph_chargingenergy, graph_energybytype

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# 2. Charging Infrastructure co-optimisation Package

## 2.1 Optimisation parameters configurations

### 2.1.1 Main optimisation parameters

In [2]:
# All parameters are now loaded from YAML configuration files:
#   config/run_Optimisation_Config.yaml   - run settings (schedule, fleet size, solver gap)
#   config/infrastructure_configuration.yaml       - infrastructure power and cost parameters
#
# To change parameters, edit the YAML files directly.
# If infrastructure_configurations is set to "custom" in the run config,
# the custom parameters defined there will override the predefined ones.

config_dir = os.path.join(project_root, 'config')

opt_config, cost_config, power_charge_config, run = load_opt_config(
    run_yaml   = os.path.join(config_dir, 'run_Optimisation_Config.yaml'),
    env_yaml  = os.path.join(config_dir, 'env.yaml'),
    infra_yaml = os.path.join(config_dir, 'predefined', 'infrastructure_configuration.yaml'),
)

schedule_name   = run.schedule_name
schedule_number = run.schedule_number

print('Configuration loaded successfully:')
print(f'  Schedule:      {schedule_name} (#{schedule_number})')
print(f'  Fleet size:    {run.EVs} vehicles')
print(f'  MIP Gap:       {run.MIPGap}')

Configuration loaded successfully:
  Schedule:      schedule_1 (#1)
  Fleet size:    50 vehicles
  MIP Gap:       0.025


### 2.1.2. Cost and power configuration

In the cost and power configurations above, it is possible to change the values of different paramenters, related to cost of infrastructure, energy subscription rates, route charging rates and battery power and losses.

In [3]:
# Power and battery parameters (loaded from YAML)
print('--- Chargers Power Parameters ---')
for k, v in power_charge_config.items():
    print(f'  {k}: {v}')

--- Chargers Power Parameters ---
  Charging_losses: 0.964
  Battery_Maximum_Limit: 0.8
  Battery_Minimum_Limit: 0.2
  Accumulated_Cycle_Capacity: 3500.0
  Charger_Power: {'f1': 7.4, 'f2': 50, 'f3': 150, 'f4': 350, 'route': 150}


In [4]:
# Infrastructure cost parameters (loaded from YAML)
print('--- Infrastructure Cost Parameters ---')
for k, v in cost_config.items():
    print(f'  {k}: {v}')

--- Infrastructure Cost Parameters ---
  Infrastructure_life: 20
  Discount_rate: 0.05
  Infrastructure_cost: {'f1': 61000, 'f2': 766000, 'f3': 1422000, 'f4': 2653000, 'route': 1980000}
  maintenance_cost: {'f1': 5000, 'f2': 40000, 'f3': 120000, 'f4': 240000, 'route': 120000}
  Infrastructure_subscription: 50
  Price_FixedrateDT: 0.0331
  Demand_rate: 1352
  Price_Fixedrateroute: 8.9


In [5]:
# Before calling optimisation()
for key in ['En_consumption', 'Ev_distance', 'EV_availability', 'Battery_Limitation', 'PowerRate_Limitation']:
    val = opt_config[key]
    print(f"\n{key}:")
    print(f"  Type:  {type(val).__name__}")
    print(f"  Shape: {val.shape if hasattr(val, 'shape') else 'N/A'}")
    if hasattr(val, 'head'):
        print(f"  Head:\n{val.head(3)}")
    else:
        print(f"  Value: {val}")


En_consumption:
  Type:  DataFrame
  Shape: (8784, 50)
  Head:
VehicleID             0    1    2    3    4    5    6    7    8    9   ...  \
date                                                                   ...   
2024-01-01 00:00:00  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   
2024-01-01 01:00:00  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   
2024-01-01 02:00:00  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   

VehicleID             40   41   42   43   44   45   46   47   48   49  
date                                                                   
2024-01-01 00:00:00  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
2024-01-01 01:00:00  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
2024-01-01 02:00:00  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  

[3 rows x 50 columns]

Ev_distance:
  Type:  DataFrame
  Shape: (8784, 50)
  Head:
VehicleID             0    1    2    3    4    5    6    7    8    9   ...  \
date                   

In [6]:
# Check if the index is actually datetime or strings
avail = opt_config['EV_availability']
print(f"Index type: {type(avail.index[0])}")
print(f"Index dtype: {avail.index.dtype}")

Index type: <class 'pandas._libs.tslibs.timestamps.Timestamp'>
Index dtype: datetime64[ns]


## 2.3 Run optimisation

In [ ]:
# Call the optimisation function
m, Price, EV_availability, Distance_km = optimisation(opt_config, cost_config, power_charge_config)

→ Solving model with 50 vehicles over 168 time steps (delta_t=1.0)...
Set parameter Username
Set parameter LicenseID to value 2678500
Academic license - for non-commercial use only - expires 2026-06-16
Read LP format model from file C:\Users\mcgr2\AppData\Local\Temp\tmphux62gvq.pyomo.lp
Reading time = 3.17 seconds
x1: 110258 rows, 126057 columns, 316655 nonzeros
Set parameter MIPGap to value 0.025
Set parameter LogFile to value "gurobi_log_V2.txt"
Set parameter Threads to value 16
Set parameter Seed to value 42
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) Ultra 7 165U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 16 threads

         Reduce the value of the Threads parameter to improve performance


Non-default parameters:
MIPGap  0.025
Seed  42
Threads  16

Optimize a model with 110258 rows, 126057 columns and 316655 nonzeros
Model fingerprint: 0x250a0a5c
Variable 

### Saving results

In [ ]:
# User can choose where to save results the results, CSV files

results_folder = os.path.join(project_root, 'data', 'Output', f'{schedule_name}', 'Results')
os.makedirs(results_folder, exist_ok=True)

csv_file_pathA = os.path.join(results_folder, f'{schedule_number}_Main_variables_results.csv')
csv_file_pathB = os.path.join(results_folder, f'{schedule_number}_results_summary.csv')
csv_file_pathC = os.path.join(results_folder, f'{schedule_number}_results_per_EV.csv')

In [ ]:
save_results(m, Price, EV_availability, Distance_km, csv_file_pathA, csv_file_pathB, csv_file_pathC, cost_config, power_charge_config)

## 2.4. Post-processing and visualisation

In [ ]:
# Set the path to your folder
file_pattern_mean = os.path.join(project_root, 'data', 'Output', f'{schedule_name}', 'Results', '*_results_summary.csv')

# Use glob to find the actual file
matched_files = glob(file_pattern_mean)
if not matched_files:
    raise FileNotFoundError(f"No file matches the pattern: {file_pattern_mean}")
file_path = matched_files[0]


plot_summary_table(file_path)

### Create maximum and average files

In [ ]:
# Set the path to your folder
folder_path = os.path.join(project_root, 'data', 'Output', f'{schedule_name}', 'Results')

# Find all files with the pattern *_pertime_averages.csv
filename_pattern = os.path.join(folder_path, '*_Main_variables_results.csv')

process_folder(folder_path, filename_pattern)

In [ ]:
# Set the path to your folder
folder_path = os.path.join(project_root, 'data', 'Output', f'{schedule_name}', 'Results')
file_pattern_max = os.path.join(project_root, 'data', 'Output', f'{schedule_name}', 'Results', '*_max_variable_per_step.csv')
files_pertime_max = glob(file_pattern_max)

graph1 = graph_number_of_chargers_by_schedules(folder_path, files_pertime_max)

### Graph 2 - Average charging power and average price of electricity

In [ ]:
# Set the path to your folder
file_pattern_mean = os.path.join(project_root, 'data', 'Output', f'{schedule_name}', 'Results', '*_avg_variable_per_step.csv')
# Use glob to find the actual file

matched_files = glob(file_pattern_mean)
if not matched_files:
    raise FileNotFoundError(f"No file matches the pattern: {file_pattern_mean}")
file_path = matched_files[0]
folder_path = os.path.join(project_root, 'data', 'Output', f'{schedule_name}', 'Results')

graph_chargingenergy(file_path, folder_path)

graph_energybytype(file_path, folder_path)

### Graph 3 - Charging, discharging power and SOC

In [ ]:
# Set the path to your folder
file_path = csv_file_pathC

# Set the number of vehicles for the graph
n_vehicles = 3
# Set the number of days for the graph
n_days = 4

graph_vehicles(folder_path, file_path, n_days, n_vehicles)